In [ ]:
!pip install --upgrade transformers accelerate datasets sentencepiece sacrebleu torch

### Retrieve the training Dataset

In [ ]:
import random
from datasets import Dataset


hsb_sentences = open("../../data/train/train.de-hsb.hsb").read().splitlines()
de_sentences = open("../../data/train/train.de-hsb.de").read().splitlines()



# Combine them into pairs
data = list(zip(de_sentences, hsb_sentences))

# Shuffle the pairs
random.shuffle(data)

# Unzip them back
de_sentences, hsb_sentences = zip(*data)

In [ ]:
data = Dataset.from_dict({
    "hsb": hsb_sentences[:-1000],
    "de": de_sentences[:-1000]
})
print(len(hsb_sentences), len(de_sentences))

### Load the tokenizer

In [ ]:
from transformers import NllbTokenizer

model_name = "facebook/nllb-200-distilled-600M"
tokenizer = NllbTokenizer.from_pretrained(model_name)

src_lang = "de_Latn"
tgt_lang = "hsb_Latn"

### Add the Upper Sorbian token to the tokenizer

In [ ]:
tokenizer.add_special_tokens({"additional_special_tokens": ["hsb_Latn"]})
lang_id = tokenizer.convert_tokens_to_ids("hsb_Latn")
print("New language ID:", lang_id)

### Retrieve the model and copy the czech embeddings

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from accelerate import init_empty_weights, dispatch_model
import torch
from accelerate import load_checkpoint_and_dispatch

def get_device_map() -> str:
    return 'cuda' if torch.cuda.is_available() else 'cpu'

device = get_device_map()

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map=device
)

model.resize_token_embeddings(len(tokenizer))


# Get Czech token ID (NLLB should have it)
czech_token_id = tokenizer.convert_tokens_to_ids("ces_Latn")  # or "__ces__"
print(f"Czech token ID: {czech_token_id}")

# Get Czech language embedding
czech_embedding = model.model.shared.weight.data[czech_token_id].clone()

# Get HSB token ID  
hsb_token_id = tokenizer.convert_tokens_to_ids("hsb_Latn")
print(f"HSB token ID: {hsb_token_id}")

# COPY Czech embedding to HSB
model.model.shared.weight.data[hsb_token_id] = czech_embedding
print(f"Copied czech → HSB language embedding!")

### Tokenize the training Dataset

In [ ]:
# set languages
tokenizer.src_lang = src_lang
tokenizer.tgt_lang = tgt_lang

def preprocess_function(examples):
    inputs = examples["de"]   # source language
    targets = examples["hsb"] # target language

    model_inputs = tokenizer(
        inputs,
        text_target=targets,  # tokenizer will handle the target separately
        max_length=256,
        truncation=True,
        padding="max_length"
    )
    return model_inputs

# tokenize the dataset
tokenized_dataset = data.map(preprocess_function, batched=True)

In [ ]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Current device: {torch.cuda.current_device()}")
print(f"Device name: {torch.cuda.get_device_name(0)}")

# Check where the model is
print(f"Model device: {model.device}")
print(f"Model is on GPU: {next(model.parameters()).is_cuda}")

### Pick the validation Dataset and set the training arguments

In [ ]:
!pip install transformers[torch]
from transformers import Seq2SeqTrainingArguments
import os


os.environ["WORLD_SIZE"] = "1"
os.environ["MASTER_ADDR"] = "127.0.0.1"
os.environ["MASTER_PORT"] = "29500"

hsb = open("dev.de-hsb.hsb").read().splitlines()
de = open("dev.de-hsb.de").read().splitlines()

eval_dataset = Dataset.from_dict({
    "hsb": hsb_sentences[-1000:],
    "de": de_sentences[-1000:]
})


training_args = Seq2SeqTrainingArguments(
    output_dir="../../models/nllb-finetuned-1",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=5e-6,
    weight_decay=0.01,
    save_total_limit=5,
    num_train_epochs=10,
    bf16= True,          # GPU if available
    
    eval_strategy = "steps",
    save_steps = 500,
    eval_steps= 500,
    predict_with_generate=True,
    generation_max_length= 128,

    load_best_model_at_end=True,
    metric_for_best_model="chrfpp",
    greater_is_better=True,
    
    logging_steps=100,
    logging_strategy="steps",
    report_to=[],       # disables WandB/other loggers
    local_rank=-1,      # disables distributed
)

### Define the compute metrics function

In [ ]:
import numpy as np
import sacrebleu


# define the compute metrics function
def compute_metrics(eval_preds):
    preds, labels = eval_preds

    # Replace -100 (ignored tokens)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Strip whitespace
    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    bleu = sacrebleu.corpus_bleu(decoded_preds, [decoded_labels])
    chrf = sacrebleu.corpus_chrf(decoded_preds, [decoded_labels], word_order=2)

    return {
        "bleu": bleu.score,
        "chrfpp": chrf.score
    }

### Tokenize the validation dataset and prepare the trainer

In [ ]:
from transformers import Seq2SeqTrainer

eval_dataset = eval_dataset.map(preprocess_function, batched=True)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

### Train the model

In [ ]:
trainer.train()